In [14]:
# ------------------------------------------------------------
# CARGA INTERACTIVA EN EL KERNEL (Colab)
# ------------------------------------------------------------
from google.colab import files
import pandas as pd
import numpy as np

print("📤 Sube el CSV de seguros:")
up_seguros = files.upload()
seguros_name = list(up_seguros.keys())[0]
df_seguros = pd.read_csv(seguros_name, dtype=str, keep_default_na=False)
print(f"✅ Seguros: {df_seguros.shape}")

print("\n📤 Sube el CSV de Alemania:")
up_alemania = files.upload()
alemania_name = list(up_alemania.keys())[0]
df_alemania = pd.read_csv(alemania_name, dtype=str, keep_default_na=False)
print(f"✅ Alemania: {df_alemania.shape}")

📤 Sube el CSV de seguros:


Saving insurance_policies_claims_2018_2025.csv to insurance_policies_claims_2018_2025 (2).csv
✅ Seguros: (52676, 76)

📤 Sube el CSV de Alemania:


Saving germany_municipalities_jun_2025.csv to germany_municipalities_jun_2025 (2).csv
✅ Alemania: (10950, 13)


In [26]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# TABLA ALEMANIA — Limpieza completa paso a paso
# ------------------------------------------------------------
# 🌍 Este dataset tiene información de todos los municipios de
#    Alemania: nombres, códigos postales, población, coordenadas...
#    Es como un directorio telefónico gigante del país, pero con
#    datos de cada pueblo y ciudad. Vamos a dejarlo limpio y listo
#    para trabajar, paso a paso y con semáforos para ver qué está bien.
print("-" * 60)
print("INICIO LIMPIEZA: germany_municipalities")
print("-" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
# 📂 Abrimos el CSV. Este archivo tiene un detalle: usa encoding
#    "utf-8-sig", que es como un UTF-8 pero con un "sello" al inicio
#    (BOM) que meten algunos programas de Windows. Si no lo usamos,
#    la primera columna aparecería con caracteres basura al inicio.
# Igual que antes: leemos todo como texto y no dejamos que Pandas
# invente nulos por su cuenta. Primero ver el archivo tal cual.
df_de = pd.read_csv(
    "germany_municipalities_jun_2025.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)
print(f"\n[1] 📥 Cargado: {df_de.shape[0]} filas x {df_de.shape[1]} columnas")
print(f"    → Es como abrir un directorio telefónico de todo un país:")
print(f"      {df_de.shape[0]:,} municipios (filas), cada uno con {df_de.shape[1]} datos (columnas).")
print(f"    Columnas: {df_de.columns.tolist()}")

# - Guardamos los nombres originales para después poder auditar
#   qué se renombró y qué se eliminó. Es como guardar la foto
#   del "antes" para compararla con el "después" al final.
cols_originales = df_de.columns.tolist()

# ------------------------------------------------------------
# 2. RENAME
# ------------------------------------------------------------
# 🏷️ Los nombres de las columnas vienen con paréntesis y mezcla de
#    inglés/alemán: "municipality_name_(gemeindename)". Eso es como
#    una etiqueta escrita por dos personas distintas: funciona, pero
#    es incómoda. Las renombramos a algo más limpio y en un solo idioma.
df_de = df_de.rename(columns={
    "municipality_name_(gemeindename)": "municipality_name",
    "municipality_designation": "municipality_type",
    "regional_designation": "regional_type",
    "per_km2": "population_density",
    "state_(land)": "state",
    "district_(kreis)": "district",
})
print(f"\n[2] 🏷️ Rename hecho. Columnas nuevas: {df_de.columns.tolist()}")
print(f"    → 'municipality_name_(gemeindename)' ahora se llama 'municipality_name'.")
print(f"      Más corto, más claro, más fácil de escribir en el código.")

# ------------------------------------------------------------
# 2.5 DIAGNÓSTICO DE NULOS Y VACÍOS
# ------------------------------------------------------------
# 🩺 Antes de operar, diagnosticamos. Contamos cuántos vacíos ("")
#    y cuántos nulos reales (NaN) hay en cada columna, y lo mostramos
#    ordenado de la más problemática a la más limpia.
print("\n" + "-" * 60)
print("[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS")
print("-" * 60)

reporte_nulos = pd.DataFrame({
    "tipo_dato":       df_de.dtypes,
    "vacios_str":      (df_de == "").sum(),
    "nan_reales":      df_de.isnull().sum(),
    "total_problemas": ((df_de == "") | df_de.isnull()).sum(),
})
reporte_nulos["pct_del_total"] = (
    reporte_nulos["total_problemas"] / len(df_de) * 100
).round(2)
reporte_nulos = reporte_nulos.sort_values("total_problemas", ascending=False)
print(reporte_nulos.to_string())

# 🚦 Y ahora el semáforo: un color por columna según qué tan grave es.
#    Es como el tablero de un auto: verde = todo bien, rojo = peligro.
print("\nSemáforo por columna:")
for col in reporte_nulos.index:
    pct = reporte_nulos.loc[col, "pct_del_total"]
    if   pct == 0:   icono = "✅"  # Perfecto, sin problemas.
    elif pct < 5:    icono = "🟢"  # Casi limpio, muy poquitos vacíos.
    elif pct < 30:   icono = "🟡"  # Ojo, hay que prestarle atención.
    elif pct < 70:   icono = "🟠"  # Bastante sucio, hay que trabajarlo.
    else:            icono = "🔴"  # Casi vacío, cuidado con esta columna.
    print(f"  {icono}  {col:<25} → {pct:>6.2f}%  "
          f"({reporte_nulos.loc[col, 'total_problemas']} filas)")

# Ahora sí, convertimos los vacíos en nulos reales. Es como marcar
# con resaltador las casillas en blanco, para encontrarlas fácil.
df_de = df_de.replace({"": np.nan})
print("\n[2.5] 🔄 Vacíos convertidos a NaN reales.")

# ------------------------------------------------------------
# 2.75 ESTANDARIZAR TEXTO
# ------------------------------------------------------------
# 🔤 Los textos (estados, distritos, nombres de municipios) vienen
#    con mayúsculas, minúsculas, espacios extra y hasta comillas.
#    Es como si cada quien hubiera escrito las etiquetas a su manera.
#    Los unificamos para poder agrupar y comparar sin sorpresas.
print("\n" + "-" * 60)
print("[2.75] 🔤 ESTANDARIZACIÓN DE TEXTO")
print("-" * 60)

text_cols = [
    "state",
    "district",
    "regional_type",
    "municipality_name",
    "municipality_type",
]

# Primero nos aseguramos de que sean texto puro (por si algún número
# se coló como texto en algún lado). Es como asegurarnos de tener
# todas las piezas del mismo material antes de pintarlas.
for col in text_cols:
    df_de[col] = df_de[col].astype(str)

# ✂️ A minúsculas y sin espacios en las orillas: "  Bavaria " → "bavaria".
for col in text_cols:
    df_de[col] = df_de[col].str.lower().str.strip()

# 🧹 Quitamos las comillas de las orillas: '"berlin"' → 'berlin'.
#    Es como pelar una naranja antes de comérsela.
for col in text_cols:
    df_de[col] = df_de[col].str.strip('"')

# 🧽 Colapsamos espacios dobles: "baden   wurttemberg" → "baden wurttemberg".
for col in text_cols:
    df_de[col] = df_de[col].str.replace(r"\s+", " ", regex=True)

# Cuando convertimos a str, los NaN se volvieron el texto "nan".
# Aquí los regresamos a nulos reales. Es como deshacer un disfraz.
df_de[text_cols] = df_de[text_cols].replace({"nan": np.nan, "": np.nan})

print("Estandarización aplicada a:")
for col in text_cols:
    print(f"  · {col}")

print(f"\nEjemplo 'state' únicos: {df_de['state'].dropna().unique()[:5]}")
print(f"Ejemplo 'district' únicos: {df_de['district'].dropna().unique()[:5]}")
print(f"  → Antes había 'Bavaria', 'BAVARIA', '  bavaria'; ahora todos son 'bavaria'.")

# ------------------------------------------------------------
# 3. NULOS LÓGICOS  VÁLIDOS
# ------------------------------------------------------------
# 🤔 Aquí un matiz importante: no todos los nulos son errores.
#    Un municipio sin "tipo" probablemente ES un municipio normal,
#    solo que nadie lo clasificó explícitamente. Igual con el
#    "regional_type": si no tiene designación, es que no aplica.
antes_nulos_mt = df_de["municipality_type"].isnull().sum()
antes_nulos_rt = df_de["regional_type"].isnull().sum()

df_de["municipality_type"] = df_de["municipality_type"].fillna("Municipio")
df_de["regional_type"]     = df_de["regional_type"].fillna("Sin Designación")

print(f"\n[3] 🤔 Nulos lógicos rellenados:")
print(f"    municipality_type: {antes_nulos_mt} filas → 'Municipio'")
print(f"    regional_type:     {antes_nulos_rt} filas → 'Sin Designación'")
print(f"  → Es como cuando en un formulario dejas 'tipo de documento' en blanco:")
print(f"    se asume que es el más común. Aquí, el tipo más común es 'Municipio'.")

# ------------------------------------------------------------
# 4. BUGS CRÍTICOS → dropna
# ------------------------------------------------------------
# 🫀 Hay datos que son como el "corazón" de una fila: si faltan,
#    la fila pierde sentido. Sin código postal o sin coordenadas,
#    un municipio no nos sirve para nada geográfico. Los eliminamos.
antes = len(df_de)
criticas = ["postal_code", "longitude", "latitude"]
print(f"\n[4] 🫀 Nulos en columnas críticas antes del dropna:")
print(df_de[criticas].isnull().sum().to_string())

df_de = df_de.dropna(subset=criticas)
print(f"    📉 Filas eliminadas: {antes - len(df_de)}")
print(f"    ✅ Filas restantes:  {len(df_de)}")
print(f"  → Es como tirar direcciones incompletas: si no hay código postal")
print(f"    ni coordenadas, el municipio no se puede ubicar en el mapa.")

# ------------------------------------------------------------
# 5. PARSEAR COORDENADAS
# ------------------------------------------------------------
# 🗺️ Las coordenadas vienen como texto, a veces con coma decimal
#    europea ("13,405" en vez de "13.405") y a veces con valores
#    rotos como "1340500" (un decimal perdido). Nuestra función
#    limpia todo eso: convierte comas en puntos y, si el número
#    es demasiado grande para ser una coordenada real, lo divide
#    entre 10 hasta que quede en el rango válido.
def parse_coord(value, tipo):
    if pd.isna(value) or str(value).strip() == "":
        return np.nan
    s = str(value).strip().replace(",", ".")
    try:
        num = float(s)
    except ValueError:
        return np.nan
    # Si un decimal se perdió al exportar, el número queda enorme.
    # Dividir entre 10 hasta que encaje en el rango correcto
    # es como "bajarle el volumen" al número hasta que suene bien.
    # Los topes son los de Alemania (lon 5.5–15.5, lat 47–55.5): con el
    # tope global de 180, "943751" quedaba en 94.3751 en vez de 9.43751.
    if tipo == "longitude":
        while num > 20:
            num /= 10
    else:
        while num > 60:
            num /= 10
    return num

df_de["longitude"] = df_de["longitude"].apply(lambda v: parse_coord(v, "longitude"))
df_de["latitude"]  = df_de["latitude"].apply(lambda v: parse_coord(v, "latitude"))

coords_rotas = df_de["longitude"].isnull().sum() + df_de["latitude"].isnull().sum()
print(f"\n[5] 🗺️ Coordenadas parseadas")
print(f"    Longitud  → min: {df_de['longitude'].min():.4f} | max: {df_de['longitude'].max():.4f}")
print(f"    Latitud   → min: {df_de['latitude'].min():.4f} | max: {df_de['latitude'].max():.4f}")
print(f"    ⚠️  Coords que no se pudieron parsear: {coords_rotas}")
print(f"  → Alemania está más o menos entre longitud 5.5 y 15.5, y latitud")
print(f"    47 y 55.5. Si los min/max caen dentro de esos rangos, todo bien.")

# ------------------------------------------------------------
# 6. CONVERTIR NUMÉRICOS
# ------------------------------------------------------------
# 🔢 Estas columnas deben ser números, pero llegaron como texto
#    porque al inicio le dijimos a Pandas que leyera todo como str.
#    Ahora las convertimos a número. Si alguna celda tiene texto
#    basura que no se puede convertir, queda como nulo (coerce).
for col in ["area_km2", "population_total", "male", "female"]:
    antes_nulos = df_de[col].isnull().sum()
    df_de[col] = pd.to_numeric(df_de[col], errors="coerce")
    despues_nulos = df_de[col].isnull().sum()
    print(f"\n[6] 🔢 {col}: {antes_nulos} → {despues_nulos} nulos tras to_numeric")
print(f"  → Si algún nulo apareció de la nada, es porque esa celda tenía")
print(f"    texto que no se podía leer como número. Mejor perder ese dato")
print(f"    que arrastrar un valor mal calculado.")

# ------------------------------------------------------------
# 7. POBLACIÓN según tipo de municipio
# ------------------------------------------------------------
# 💡 Hay municipios especiales en Alemania llamados "gemeindefreies
#    Gebiet" (área libre de municipio): son zonas sin población real,
#    como bosques o terrenos militares. Para esos, la población
#    nula ES correcta: debería ser 0. No son datos faltantes.
mask_no_habitado = df_de["municipality_type"].str.contains("gemfr", case=False, na=False)
print(f"\n[7] 💡 Municipios 'gemfr. gebiet' (sin población real): {mask_no_habitado.sum()}")
print(f"  → Son áreas deshabitadas por definición, no errores del sistema.")

# Solo miramos los habitados: esos sí deberían tener población.
mask_habitado = ~mask_no_habitado
poblacion_nula_habitado = df_de.loc[mask_habitado, "population_total"].isnull().sum()
print(f"    Municipios habitados con población nula: {poblacion_nula_habitado}")

# Rellenamos con 0 solo los no habitados (correcto).
df_de.loc[mask_no_habitado, "population_total"] = (
    df_de.loc[mask_no_habitado, "population_total"].fillna(0)
)
# Y eliminamos los habitados con población nula: ahí sí hay un error.
antes = len(df_de)
df_de = df_de.dropna(subset=["population_total"])
print(f"    📉 Filas eliminadas por población nula: {antes - len(df_de)}")

# ------------------------------------------------------------
# 8. DENSIDAD recalculada + drop de area_km2
# ------------------------------------------------------------
# 🧮 La columna "per_km2" (densidad original) puede venir mal calculada
#    o traer valores raros. Mejor la recalculamos nosotros mismos a
#    partir de población y área. Es como verificar una cuenta ajena
#    con calculadora propia.
#
# ⚠️ Ojo con la división por cero: si algún municipio tiene área 0
#    (probablemente un error), la división daría infinito. Lo tratamos.
area_cero = (df_de["area_km2"] == 0).sum()
print(f"\n[8] 🧮 Densidad recalculada:")
print(f"    Filas con area_km2 = 0 (dividirían por 0): {area_cero}")

df_de["population_density"] = df_de["population_total"] / df_de["area_km2"]
# Los infinitos se convierten en nulos: mejor un hueco que un valor absurdo.
df_de["population_density"] = df_de["population_density"].replace(
    [np.inf, -np.inf], np.nan
)

# Después de recalcular la densidad, ya no necesitamos area_km2.
# Es como usar una escalera para subir al techo y luego tirarla:
# ya cumplió su función, no hay por qué cargarla.
df_de = df_de.drop(columns=["area_km2"])
print(f"    ✂️ Columna 'area_km2' eliminada.")
print(f"  → La usamos para calcular la densidad, y ahora ya no la necesitamos.")

# ------------------------------------------------------------
# 9. AUDITORÍA FINAL — comparativa antes/después
# ------------------------------------------------------------
# 🧾 Repaso final antes de guardar. Como el chef que prueba el plato
#    antes de sacarlo al cliente: verificamos que todo esté en orden.
print("\n" + "-" * 60)
print("🧾 AUDITORÍA FINAL")
print("-" * 60)
print(f"Filas finales: {len(df_de):,}")
print(f"\n🔍 Nulos restantes por columna:")
print(df_de.isnull().sum().sort_values(ascending=False))
print(f"\n🔑 Duplicados: {df_de.duplicated().sum()}")
print(f"  → Si es 0, cada fila es única. Si es >0, hay que investigar.")
print(f"\n👀 Primeras 3 filas:")
print(df_de.head(3).to_string())
print(f"\n📋 Tipos de datos:")
print(df_de.dtypes)

# ------------------------------------------------------------
# 10. GUARDAR
# ------------------------------------------------------------
# 💾 Guardamos todo limpio en un CSV nuevo. Es como sacar una foto
#    del trabajo terminado para no repetir la limpieza después.
df_de.to_csv("germany_municipalities_clean.csv", index=False, encoding="utf-8")
print(f"\n✅ Guardado: germany_municipalities_clean.csv ({len(df_de)} filas)")

# ------------------------------------------------------------
# 11. VALIDACIONES POST-GUARDADO
# ------------------------------------------------------------
# 📸 Y ahora la parte más importante: volvemos a abrir el archivo
#    que acabamos de guardar y lo revisamos todo de nuevo.
#    Es como mirar la foto que acabamos de tomar por si salió borrosa.
print("\n" + "=" * 60)
print("🔍 VALIDACIONES POST-GUARDADO")
print("=" * 60)

# - Recargar el CSV guardado para verificar integridad
df_check = pd.read_csv("germany_municipalities_clean.csv")

# ------------------------------------------------------------
# 11.1 Resumen general
# ------------------------------------------------------------
print(f"\n[11.1] 📊 RESUMEN GENERAL")
print(f"    Filas guardadas:    {df_check.shape[0]:,}")
print(f"    Columnas guardadas: {df_check.shape[1]}")
print(f"    Tamaño en memoria:  {df_check.memory_usage(deep=True).sum() / 1024:.2f} KB")
print(f"  → Si las filas coinciden con las que teníamos antes de guardar,")
print(f"    el archivo se escribió bien. Si no, algo se perdió en el camino.")

# ------------------------------------------------------------
# 11.2 Columnas finales (orden exacto)
# ------------------------------------------------------------
print(f"\n[11.2] 📜 COLUMNAS FINALES ({df_check.shape[1]}):")
for i, col in enumerate(df_check.columns, 1):
    print(f"    {i:>2}. {col:<25} ({df_check[col].dtype})")
print(f"  → Este es el inventario exacto de lo que hay en la bodega final.")

# ------------------------------------------------------------
# 11.3 Columnas eliminadas
# ------------------------------------------------------------
print(f"\n[11.3] ✂️ COLUMNAS ELIMINADAS:")
# Aquí comparamos lo original con lo final, ajustando por los renombres:
# una columna renombrada no es "eliminada", solo cambió de nombre.
cols_renombradas = {
    "municipality_name_(gemeindename)": "municipality_name",
    "municipality_designation": "municipality_type",
    "regional_designation": "regional_type",
    "per_km2": "population_density",
    "state_(land)": "state",
    "district_(kreis)": "district",
}

# - Ajustando las que fueron renombradas (no son "eliminadas")
cols_originales_efectivas = [
    cols_renombradas.get(c, c) for c in cols_originales
]
cols_eliminadas = [
    c for c in cols_originales_efectivas if c not in df_check.columns
]

if cols_eliminadas:
    for col in cols_eliminadas:
        print(f"    ✗ {col}")
else:
    print("    (ninguna)")

print(f"\n    Total columnas originales:  {len(cols_originales)}")
print(f"    Total columnas renombradas: {len(cols_renombradas)}")
print(f"    Total columnas eliminadas:  {len(cols_eliminadas)}")
print(f"    Total columnas finales:     {len(df_check.columns)}")
print(f"  → Si renombradas + eliminadas + conservadas = originales,")
print(f"    no se nos escapó ninguna columna por el camino.")

# ------------------------------------------------------------
# 11.4 Verificación de nulos restantes
# ------------------------------------------------------------
print(f"\n[11.4] 🔍 NULOS EN EL CSV GUARDADO:")
nulos_check = df_check.isnull().sum()
nulos_check = nulos_check[nulos_check > 0].sort_values(ascending=False)

if len(nulos_check) == 0:
    print("    ✅ Sin nulos restantes")
    print("    → La tabla quedó limpia como un vidrio recién lavado.")
else:
    for col, n in nulos_check.items():
        pct = n / len(df_check) * 100
        # 🚦 Semáforo también aquí, para ver rápido qué columnas
        #    conservan huecos y cuánta gravedad tiene.
        if   pct < 5:    icono = "🟢"
        elif pct < 30:   icono = "🟡"
        elif pct < 70:   icono = "🟠"
        else:            icono = "🔴"
        print(f"    {icono} {col:<25} → {n:>6} filas ({pct:>6.2f}%)")

# ------------------------------------------------------------
# 11.5 Verificación de duplicados
# ------------------------------------------------------------
dups = df_check.duplicated().sum()
print(f"\n[11.5] 🔑 DUPLICADOS EN EL CSV GUARDADO: {dups}")
if dups == 0:
    print("    ✅ Sin duplicados")
    print("    → Cada fila es única, no hay municipios repetidos.")
else:
    print(f"    ⚠️  {dups} filas duplicadas")
    print(f"    → Habría que investigar por qué se repitieron.")

# ------------------------------------------------------------
# 11.6 Verificación de columnas que NO deben estar
# ------------------------------------------------------------
print(f"\n[11.6] 🚫 VERIFICACIÓN DE COLUMNAS ELIMINADAS:")
# Esta verificación es como revisar que la basura que sacamos
# efectivamente NO se quedó dentro de la casa.
cols_prohibidas = ["area_km2", "per_km2", "municipality_name_(gemeindename)",
                   "municipality_designation", "regional_designation",
                   "state_(land)", "district_(kreis)"]
encontradas = [c for c in cols_prohibidas if c in df_check.columns]
if encontradas:
    print(f"    ⚠️  Columnas que NO deberían estar: {encontradas}")
else:
    print("    ✅ Ninguna columna eliminada/renombrada quedó en el CSV")
    print("    → La limpieza fue efectiva: lo que sacamos, se quedó fuera.")

# ------------------------------------------------------------
# 11.7 Verificación de tipos de datos
# ------------------------------------------------------------
print(f"\n[11.7] 📋 TIPOS DE DATOS:")
print(df_check.dtypes.to_string())
print(f"  → Los tipos numéricos deberían ser float/int, y los textos object.")
print(f"    Si algún número aparece como object, hay texto escondido adentro.")

# ------------------------------------------------------------
# 11.8 Verificación de rangos de coordenadas
# ------------------------------------------------------------
# 🗺️ Verificación geográfica: si las coordenadas caen todas dentro
#    del "rectángulo" de Alemania, todo tiene sentido. Si alguna
#    cae en el océano o en otro país, algo se rompió al parsear.
print(f"\n[11.8] 🗺️ RANGOS DE COORDENADAS:")
if "longitude" in df_check.columns and "latitude" in df_check.columns:
    print(f"    Longitud  → min: {df_check['longitude'].min():>8.4f} | "
          f"max: {df_check['longitude'].max():>8.4f}")
    print(f"    Latitud   → min: {df_check['latitude'].min():>8.4f} | "
          f"max: {df_check['latitude'].max():>8.4f}")

    # - Validar rango geográfico de Alemania (aprox.)
    lon_ok = df_check["longitude"].between(5.5, 15.5).all()
    lat_ok = df_check["latitude"].between(47.0, 55.5).all()
    print(f"    ¿Longitudes dentro de Alemania (5.5–15.5)?:  {lon_ok}")
    print(f"    ¿Latitudes dentro de Alemania (47.0–55.5)?: {lat_ok}")
    if lon_ok and lat_ok:
        print(f"  → ✅ Todas las coordenadas caen dentro de Alemania.")
        print(f"    Ningún municipio apareció flotando en el Atlántico.")
    else:
        print(f"  → ⚠️  Hay coordenadas fuera de Alemania: hay que revisar el parseo.")
else:
    print("    ⚠️  Columnas de coordenadas no encontradas")

# ------------------------------------------------------------
# 11.9 Verificación de densidad
# ------------------------------------------------------------
# 📊 La densidad de población debería ser un número razonable:
#    ningún municipio tiene millones de personas por km², ni
#    debería haber densidades negativas (sería absurdo).
print(f"\n[11.9] 📊 VERIFICACIÓN DE DENSIDAD:")
if "population_density" in df_check.columns:
    print(f"    Densidad → min: {df_check['population_density'].min():>10.2f} | "
          f"max: {df_check['population_density'].max():>10.2f}")
    print(f"    Densidad → media: {df_check['population_density'].mean():>10.2f}")
    nulos_dens = df_check["population_density"].isnull().sum()
    print(f"    Densidad nula: {nulos_dens} filas")
    if nulos_dens > 0:
        print(f"    → Esos son los municipios con area_km2 = 0 (división por cero).")

# ------------------------------------------------------------
# 11.10 Verificación de población
# ------------------------------------------------------------
# 🇩🇪 La población total de Alemania ronda los 83 millones de personas.
#    Si la suma se aleja mucho de ese número, algo se rompió al
#    convertir los datos. También verificamos que male + female
#    sea aproximadamente igual a population_total.
print(f"\n[11.10] 🇩🇪 VERIFICACIÓN DE POBLACIÓN:")
if "population_total" in df_check.columns:
    print(f"    Población total del país: {df_check['population_total'].sum():,.0f}")
    print(f"    → Alemania tiene ~83 millones de habitantes.")
    print(f"    Población → min: {df_check['population_total'].min():>10.0f} | "
          f"max: {df_check['population_total'].max():>10.0f}")
    if "male" in df_check.columns and "female" in df_check.columns:
        suma_mf = df_check["male"].sum() + df_check["female"].sum()
        diff = abs(suma_mf - df_check["population_total"].sum())
        print(f"    Suma male + female: {suma_mf:,.0f}")
        print(f"    Diferencia vs population_total: {diff:,.0f}")
        print(f"    → Si la diferencia es pequeña, los subtotales cuadran")
        print(f"      con el total. Si es enorme, hay un error de captura.")

# ------------------------------------------------------------
# 11.11 Verificación de estados únicos
# ------------------------------------------------------------
# 🗾 Alemania tiene 16 estados federados (Bundesländer). Si después
#    de limpiar hay 16 estados únicos, todo cuadra. Si hay más, es
#    porque se colaron variantes mal escritas.
print(f"\n[11.11] 🗾 ESTADOS ÚNICOS (state):")
if "state" in df_check.columns:
    estados = df_check["state"].dropna().unique()
    print(f"    Total estados: {len(estados)}")
    print(f"    → Alemania tiene 16 estados federados. Si aquí aparecen más,")
    print(f"      es que hay duplicados con ortografía distinta.")
    for e in sorted(estados):
        print(f"      · {e}")

# ------------------------------------------------------------
# 11.12 Resultado final
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("✅ VALIDACIONES COMPLETADAS")
print("=" * 60)
print(f"📁 Archivo: germany_municipalities_clean.csv")
print(f"📊 Filas:   {df_check.shape[0]:,}")
print(f"📋 Columnas: {df_check.shape[1]}")
print(f"✂️  Eliminadas: {len(cols_eliminadas)} columnas")
print("=" * 60)
print(f"→ Con esto, la tabla de municipios alemanes quedó lista")
print(f"  para el análisis: sin coordenadas rotas, sin textos repetidos,")
print(f"  y con la densidad recalculada en base a datos confiables.")

------------------------------------------------------------
INICIO LIMPIEZA: germany_municipalities
------------------------------------------------------------

[1] 📥 Cargado: 10950 filas x 13 columnas
    → Es como abrir un directorio telefónico de todo un país:
      10,950 municipios (filas), cada uno con 13 datos (columnas).
    Columnas: ['state_(land)', 'district_(kreis)', 'regional_designation', 'municipality_name_(gemeindename)', 'municipality_designation', 'area_km2', 'population_total', 'male', 'female', 'per_km2', 'postal_code', 'longitude', 'latitude']

[2] 🏷️ Rename hecho. Columnas nuevas: ['state', 'district', 'regional_type', 'municipality_name', 'municipality_type', 'area_km2', 'population_total', 'male', 'female', 'population_density', 'postal_code', 'longitude', 'latitude']
    → 'municipality_name_(gemeindename)' ahora se llama 'municipality_name'.
      Más corto, más claro, más fácil de escribir en el código.

-----------------------------------------------------

In [37]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# LIMPIEZA TABLA DE SEGUROS (PRE-MERGE)
# Objetivo: dejar la tabla lista para el merge con Alemania
# Problema: caída del 20% en servicio de pólizas
# Versión: COMPLETA Y CORREGIDA
# ------------------------------------------------------------
print("=" * 60)
print("LIMPIEZA PRE-MERGE: insurance_policies_claims")
print("=" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
df_in = pd.read_csv(
    "insurance_policies_claims_2018_2025.csv",
    dtype=str,
    keep_default_na=False,
)
print(f"\n[1] 📥 Cargado: {df_in.shape[0]} filas x {df_in.shape[1]} columnas")

# ------------------------------------------------------------
# 1.b VALIDACIÓN TEMPRANA
# ------------------------------------------------------------
if len(df_in) == 0:
    raise ValueError("❌ El DataFrame está vacío al cargar")
print(f"    ✅ DataFrame no vacío: {len(df_in):,} filas")


# ------------------------------------------------------------
# Helper: división segura
# ------------------------------------------------------------
def safe_pct(numerador, denominador):
    """División segura que retorna 0 si el denominador es 0."""
    return numerador / denominador if denominador > 0 else 0


# ------------------------------------------------------------
# 2. DROP columnas innecesarias
# ------------------------------------------------------------
cols_a_eliminar = ["customer_email"]
cols_presentes = [c for c in cols_a_eliminar if c in df_in.columns]
if cols_presentes:
    df_in = df_in.drop(columns=cols_presentes)
    print(f"\n[2] ✂️ Columnas eliminadas: {cols_presentes}")

# ------------------------------------------------------------
# 2.5 DIAGNÓSTICO DE NULOS
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS")
print("-" * 60)

reporte_nulos = pd.DataFrame({
    "tipo_dato":       df_in.dtypes,
    "vacios_str":      (df_in == "").sum(),
    "total_problemas": ((df_in == "") | df_in.isnull()).sum(),
})
reporte_nulos["pct_del_total"] = (
    reporte_nulos["total_problemas"] / len(df_in) * 100
).round(2)
reporte_nulos = reporte_nulos.sort_values("total_problemas", ascending=False)
print(reporte_nulos[reporte_nulos["total_problemas"] > 0].to_string())

df_in = df_in.replace({"": np.nan})

# ------------------------------------------------------------
# 2.6 DIAGNÓSTICO DE NEGATIVOS
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.6] 💸 DIAGNÓSTICO DE VALORES NEGATIVOS")
print("-" * 60)

cols_montos = [c for c in df_in.columns if c.endswith("_eur")]
for col in cols_montos:
    serie = pd.to_numeric(df_in[col], errors="coerce")
    n_neg = (serie < 0).sum()
    if n_neg > 0:
        print(f"    ⚠️  {col:<32} → {n_neg:,} negativos "
              f"(min={serie.min():,.2f})")
    else:
        print(f"    ✅ {col:<32} → sin negativos")

# ------------------------------------------------------------
# 2.75 ESTANDARIZACIÓN DE TEXTO
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.75] 🔤 ESTANDARIZACIÓN DE TEXTO")
print("-" * 60)

# 2.75.1 Fechas
date_columns = [
    'date_of_birth', 'policy_start_date', 'policy_end_date',
    'renewal_date', 'claim_date', 'incident_date',
    'adjustment_date', 'settlement_date', 'created_at', 'updated_at'
]
# El CSV mezcla varios formatos en la MISMA columna:
#   2025-04-02 | 2025/04/02 | 02-04-2025 | 02 Apr 2025 | 02/04/2025 o 04/02/2025
# dayfirst=True + format='mixed' confundía día y mes en las fechas ISO
# (2025-04-02 → 2 de abril se leía como 4 de febrero). Aquí cada formato
# se lee con su propio patrón, sin adivinar.
n_ambiguas = 0

def parsear_fechas(serie):
    global n_ambiguas
    s = serie.astype(str).str.strip()
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    def aplicar(mask, fmt, texto=None):
        if mask.any():
            t = s[mask] if texto is None else texto[mask]
            out.loc[mask] = pd.to_datetime(t, format=fmt, errors="coerce")

    # AAAA-MM-DD (con o sin hora)
    aplicar(s.str.match(r"^\d{4}-\d{2}-\d{2}"), "ISO8601")
    # AAAA/MM/DD
    m = s.str.match(r"^\d{4}/\d{2}/\d{2}$")
    aplicar(m, "%Y-%m-%d", s.str.replace("/", "-", regex=False))
    # DD-MM-AAAA
    aplicar(s.str.match(r"^\d{2}-\d{2}-\d{4}$"), "%d-%m-%Y")
    # DD Mon AAAA (02 Apr 2025)
    aplicar(s.str.match(r"^\d{2} [A-Za-z]{3} \d{4}$"), "%d %b %Y")
    # xx/xx/AAAA: puede ser DD/MM o MM/DD
    m = s.str.match(r"^\d{2}/\d{2}/\d{4}$")
    if m.any():
        a = pd.to_numeric(s.str[:2].where(m), errors="coerce")
        b = pd.to_numeric(s.str[3:5].where(m), errors="coerce")
        m_dm = m & (a > 12)                 # 25/03/2024 → solo puede ser DD/MM
        m_md = m & (b > 12)                 # 03/25/2024 → solo puede ser MM/DD
        m_amb = m & ~m_dm & ~m_md           # 03/04/2024 → ambigua
        aplicar(m_dm, "%d/%m/%Y")
        aplicar(m_md, "%m/%d/%Y")
        aplicar(m_amb, "%d/%m/%Y")          # criterio: día primero (formato alemán)
        n_ambiguas += int(m_amb.sum())
    return out

for col in date_columns:
    if col in df_in.columns:
        df_in[col] = parsear_fechas(df_in[col])
print(f"    ✅ {len([c for c in date_columns if c in df_in.columns])} "
      f"columnas de fecha normalizadas")
print(f"    ⚠️  {n_ambiguas:,} fechas tipo xx/xx/AAAA son ambiguas "
      f"(día y mes ≤ 12): se interpretaron como DD/MM")

# 2.75.2 Categóricas a minúsculas
cat_cols = ["policy_status", "claim_status", "insurance_type", "coverage_type",
            "payment_frequency", "underwriting_decision"]
for col in cat_cols:
    if col in df_in.columns:
        df_in[col] = df_in[col].str.lower().str.strip()

# 2.75.3 IDs limpios
id_cols = ["policy_id", "policy_number", "customer_id", "agent_id",
           "broker_id", "adjuster_id", "claim_id", "postal_code"]
for col in id_cols:
    if col in df_in.columns:
        df_in[col] = df_in[col].str.strip()

# 2.75.4 Nombres y direcciones
name_cols = ["customer_name", "agent_name", "adjuster_name"]
for col in name_cols:
    if col in df_in.columns:
        df_in[col] = df_in[col].str.strip().str.replace(r"\s+", " ", regex=True)

# 2.75.5 Teléfonos
def normalizar_telefono(v):
    if pd.isna(v):
        return np.nan
    s = "".join(c for c in str(v).strip() if c.isdigit() or c == "+")
    return s if len(s) >= 8 else np.nan

if "customer_phone" in df_in.columns:
    df_in["customer_phone"] = df_in["customer_phone"].apply(normalizar_telefono)

# 2.75.6 Género
def normalizar_genero(v):
    if pd.isna(v):
        return np.nan
    g = str(v).strip().lower()
    if g in ("m", "male"):
        return "M"
    if g in ("f", "female"):
        return "F"
    if g in ("other", "o", "divers", "diverse", "non-binary", "nonbinary"):
        return "Other"
    if g in ("no indica", "no indica.", "n/a", "na", "unknown",
             "prefiero no decir", "prefiero no indicar", "-", ""):
        return "No indica"
    return np.nan

df_in["gender"] = df_in["gender"].apply(normalizar_genero)

# 2.75.7 País
if 'customer_country' in df_in.columns:
    country_map = {'GERMANY': 'Germany', 'Germany': 'Germany',
                   'Deutschland': 'Germany', 'DE': 'Germany'}
    df_in['customer_country'] = df_in['customer_country'].map(country_map).fillna('Germany')

# 2.75.8 Estado de póliza
status_map = {'active': 'Active', 'expired': 'Expired',
              'cancelled': 'Cancelled', 'lapsed': 'Lapsed',
              'pending': 'Pending'}
if 'policy_status' in df_in.columns:
    df_in['policy_status'] = df_in['policy_status'].map(status_map).fillna('Unknown')

# 2.75.9 Estado de siniestro
claim_status_map = {
    'paid': 'Paid', 'rejected': 'Rejected', 'denied': 'Denied',
    'open': 'Open', 'closed': 'Closed',
    'under_review': 'In Progress', 'in_progress': 'In Progress',
    'approved': 'Approved',
}
if 'claim_status' in df_in.columns:
    df_in['claim_status'] = df_in['claim_status'].map(claim_status_map).fillna('Unknown')

# 2.75.10 Frecuencia de pago
freq_map = {
    'single_payment': 'Single Payment', 'monthly': 'Monthly',
    'quarterly': 'Quarterly', 'semi_annual': 'Semi-Annual',
    'annual': 'Annual',
}
if 'payment_frequency' in df_in.columns:
    df_in['payment_frequency'] = df_in['payment_frequency'].map(freq_map).fillna('Unknown')

# 2.75.11 Tipo de seguro
ins_map = {
    'auto': 'Auto', 'home': 'Home', 'health': 'Health', 'hlth': 'Health',
    'life': 'Life', 'travel': 'Travel', 'trvl': 'Travel',
    'pet': 'Pet', 'business': 'Business', 'bus': 'Business',
    'liability': 'Liability', 'liab': 'Liability',
}
if 'insurance_type' in df_in.columns:
    df_in['insurance_type'] = df_in['insurance_type'].map(ins_map).fillna('Other')

# 2.75.12 Cobertura
cov_map = {
    'premium': 'Premium', 'basic': 'Basic', 'standard': 'Standard',
    'silver': 'Silver', 'gold': 'Gold', 'platinum': 'Platinum',
    'comprehensive': 'Comprehensive',
    'n/a': 'Not Specified', 'na': 'Not Specified',
}
if 'coverage_type' in df_in.columns:
    df_in['coverage_type'] = df_in['coverage_type'].map(cov_map).fillna('Not Specified')

# 2.75.13 Booleanos
for col in ['police_report_filed', 'complaint_filed', 'fraud_flag']:
    if col in df_in.columns:
        df_in[col] = df_in[col].map({True: True, False: False, 'True': True,
                                     'False': False, '-': None, '': None})

# 2.75.14 Underwriting decision
uw_map = {
    'approved': 'Approved', 'denied': 'Denied',
    'conditional': 'Conditional', 'referred': 'Referred',
    'pending_review': 'Pending Review', 'pending': 'Pending Review',
}
if 'underwriting_decision' in df_in.columns:
    df_in['underwriting_decision'] = df_in['underwriting_decision'].map(uw_map).fillna('Unknown')

# 2.75.15 Churn risk
churn_map = {
    'high': 'High', 'medium': 'Medium', 'low': 'Low', 'critical': 'Critical',
}
if 'churn_risk' in df_in.columns:
    df_in['churn_risk'] = df_in['churn_risk'].map(churn_map).fillna('Unknown')

# ------------------------------------------------------------
# 2.8 CONVERSIÓN NUMÉRICA
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.8] 🔢 CONVERSIÓN DE COLUMNAS NUMÉRICAS")
print("-" * 60)

cols_numericas = [
    "premium_amount_eur", "coverage_limit_eur", "deductible_eur",
    "total_paid_eur", "outstanding_balance_eur",
    "claim_amount_eur", "claim_approved_amount_eur", "settlement_amount_eur",
    "age", "fraud_score", "risk_score",
    "agent_commission_pct", "broker_commission_pct",
    "discount_applied_pct", "no_claims_bonus_pct",
    "loyalty_discount_pct", "multi_policy_discount_pct",
    "customer_lifetime_value_eur", "customer_tenure_months",
    "number_of_policies", "number_of_claims",
    "customer_satisfaction_score", "nps_score",
    "renewal_probability_pct", "commission_amount_eur",
]
convertidas = 0
for col in cols_numericas:
    if col in df_in.columns:
        df_in[col] = pd.to_numeric(df_in[col], errors="coerce")
        convertidas += 1
print(f"    ✅ {convertidas} columnas convertidas a numérico")

# ------------------------------------------------------------
# 2.85 DETECCIÓN DE NEGATIVOS
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.85] 💰 DETECCIÓN DE NEGATIVOS (sin corregir aún)")
print("-" * 60)

montos_positivos = [
    "premium_amount_eur", "coverage_limit_eur", "deductible_eur",
    "total_paid_eur", "claim_amount_eur",
    "claim_approved_amount_eur", "settlement_amount_eur",
    "customer_lifetime_value_eur",
]
for col in montos_positivos:
    if col not in df_in.columns:
        continue
    n_neg = (df_in[col] < 0).sum()
    if n_neg > 0:
        print(f"    ⚠️  {col:<32} → {n_neg:,} negativos (se corrigen en [7.96.4])")
    else:
        print(f"    ✅ {col:<32} → sin negativos")

# ------------------------------------------------------------
# 2.9 FILTRO VIDA
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[2.9] 🎯 FILTRO: SOLO SEGUROS DE VIDA")
print("-" * 60)

antes = len(df_in)
df_in = df_in[df_in["insurance_type"] == "Life"].copy()
print(f"📉 Filas eliminadas: {antes - len(df_in)}  →  Restantes: {len(df_in)}")

# ------------------------------------------------------------
# 3. DROPNA en columnas críticas
# ------------------------------------------------------------
antes = len(df_in)
df_in = df_in.dropna(subset=[
    "policy_id", "customer_id", "premium_amount_eur", "policy_start_date",
])
print(f"\n[3] 🫀 Filas eliminadas por falta de datos críticos: {antes - len(df_in)}")
print(f"    ✅ Filas restantes: {len(df_in)}")

# ------------------------------------------------------------
# 4. NULOS LÓGICOS → CENTINELAS
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[4] 🤔 TRATAMIENTO DE NULOS LÓGICOS")
print("-" * 60)

rellenos = {
    "customer_phone":  "Sin Teléfono",
    "complaint_type":  "Sin Queja",
    "broker_id":       "Sin Broker",
    "notes":           "Sin Notas",
    "gender":          "No indica",
}
for col, valor in rellenos.items():
    if col in df_in.columns:
        n = df_in[col].isnull().sum()
        df_in[col] = df_in[col].fillna(valor)
        print(f"    ✅ {col:<28} → '{valor}'  ({n} filas)")

if "outstanding_balance_eur" in df_in.columns:
    n = df_in["outstanding_balance_eur"].isnull().sum()
    df_in["outstanding_balance_eur"] = df_in["outstanding_balance_eur"].fillna(0)
    print(f"    ✅ {'outstanding_balance_eur':<28} → 0  ({n} filas)")

if "fraud_flag" in df_in.columns:
    n = df_in["fraud_flag"].isnull().sum()
    df_in["fraud_flag"] = df_in["fraud_flag"].fillna(False).astype(bool)
    print(f"    ✅ {'fraud_flag':<28} → False  ({n} filas)")

# ✅ Encuestas → NaN (NO 0)
print("\n[4.b] 📊 Encuestas: NaN explícito")
for col in ["customer_satisfaction_score", "nps_score"]:
    if col in df_in.columns:
        n_antes = df_in[col].isnull().sum()
        df_in[col] = pd.to_numeric(df_in[col], errors="coerce").replace(0, np.nan)
        n_despues = df_in[col].isnull().sum()
        print(f"    ✅ {col:<28} → NaN  "
              f"({n_antes} nulos + {n_despues - n_antes} ceros)")

df_in['survey_responded'] = df_in['customer_satisfaction_score'].notna()

# ------------------------------------------------------------
# 5. DROP COLUMNAS NO-VIDA
# ------------------------------------------------------------
cols_no_vida = [
    "vehicle_make", "vehicle_model", "vehicle_year", "vehicle_value_eur",
    "property_type", "property_value_eur", "property_year_built",
]
cols_presentes = [c for c in cols_no_vida if c in df_in.columns]
if cols_presentes:
    df_in = df_in.drop(columns=cols_presentes)
    print(f"\n[5] ✂️ Columnas no-vida eliminadas: {len(cols_presentes)}")

# ------------------------------------------------------------
# 6. SINIESTROS: clasificar
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[6] 💡 CLASIFICAR PÓLIZAS SEGÚN SINIESTRO")
print("-" * 60)

sin_claim = df_in["claim_id"].isna()
con_claim = ~sin_claim
print(f"    Pólizas SIN siniestro: {sin_claim.sum():,}")
print(f"    Pólizas CON siniestro: {con_claim.sum():,}")

# Una póliza sin siniestro no puede tener costo: el CSV trae montos
# sueltos en esas filas (ruido) que inflarían la siniestralidad. Se fuerzan a 0.
for c in ["claim_amount_eur", "claim_approved_amount_eur", "settlement_amount_eur"]:
    n_ruido = (df_in.loc[sin_claim, c].fillna(0) != 0).sum()
    df_in.loc[sin_claim, c] = 0
    print(f"    🧹 {c:<28} → 0 en pólizas sin siniestro ({n_ruido:,} filas con monto residual)")

mask_bug = con_claim & df_in["settlement_amount_eur"].isna()
print(f"    🫀 Filas eliminadas (siniestro sin settlement): {mask_bug.sum()}")
df_in = df_in[~mask_bug]

# ------------------------------------------------------------
# 6.5 NULOS LÓGICOS DE RECLAMO
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[6.5] 🔧 NULOS LÓGICOS DE RECLAMO")
print("-" * 60)

sin_claim = df_in["claim_id"].isna()
con_claim = ~sin_claim

df_in.loc[sin_claim, "claim_id"]             = "Sin Siniestro"
df_in.loc[sin_claim, "claim_description"]    = "Sin Descripción"
df_in.loc[sin_claim, "adjuster_id"]          = "Sin Asignar"
df_in.loc[sin_claim, "adjuster_name"]        = "Sin Asignar"
df_in.loc[sin_claim, "claim_denial_reason"]  = "No Aplica"
df_in.loc[sin_claim, "police_report_number"] = "Sin Reporte Policial"

df_in.loc[con_claim, "adjuster_id"]          = df_in.loc[con_claim, "adjuster_id"].fillna("Ajustador Desconocido")
df_in.loc[con_claim, "adjuster_name"]        = df_in.loc[con_claim, "adjuster_name"].fillna("Ajustador Desconocido")
df_in.loc[con_claim, "claim_description"]    = df_in.loc[con_claim, "claim_description"].fillna("Descripción Faltante")
df_in.loc[con_claim, "claim_denial_reason"]  = df_in.loc[con_claim, "claim_denial_reason"].fillna("No Especificado")
df_in.loc[con_claim, "police_report_number"] = df_in.loc[con_claim, "police_report_number"].fillna("Sin Reporte Policial")
df_in.loc[con_claim, "claim_id"]             = df_in.loc[con_claim, "claim_id"].fillna("Siniestro Desconocido")
print(f"    ✅ Sin siniestro ({sin_claim.sum():,}) → 'Sin X'")
print(f"    ⚠️  Con siniestro ({con_claim.sum():,}) → nulos marcados")

# ------------------------------------------------------------
# 6.9 RELLENO GENÉRICO
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[6.9] 🛟 RELLENO GENÉRICO")
print("-" * 60)

centinelas = {
    "claim_date": "Sin Fecha", "incident_date": "Sin Fecha",
    "incident_location": "Desconocido", "adjustment_date": "Sin Fecha",
    "settlement_date": "Sin Fecha", "customer_address": "Desconocido",
    "customer_city": "Desconocido", "customer_state": "Desconocido",
    "customer_country": "Desconocido",
}
for col, valor in centinelas.items():
    if col in df_in.columns:
        n = df_in[col].isnull().sum()
        if n > 0:
            df_in[col] = df_in[col].fillna(valor)
            print(f"    ✅ {col:<28} → '{valor}'  ({n})")

for col in df_in.select_dtypes(include=["object"]).columns:
    n = df_in[col].isnull().sum()
    if n > 0:
        df_in[col] = df_in[col].fillna("Desconocido")

for col in df_in.select_dtypes(include=[np.number]).columns:
    n = df_in[col].isnull().sum()
    if n > 0:
        df_in[col] = df_in[col].fillna(0)

# ------------------------------------------------------------
# 7. EDAD
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[7] 📊 EDAD: mediana por género")
print("-" * 60)

df_in["age"] = pd.to_numeric(df_in["age"], errors="coerce")
df_in["age"] = df_in.groupby("gender", dropna=False)["age"].transform(
    lambda s: s.fillna(s.median())
)
df_in["age"] = df_in["age"].fillna(df_in["age"].median())
print(f"    ✅ Edades nulas después: {df_in['age'].isna().sum()}")

# ------------------------------------------------------------
# 7.5 NORMALIZAR city/state PARA EL MERGE
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[7.5] 🔤 NORMALIZAR city/state PARA EL MERGE")
print("-" * 60)

if 'customer_city' in df_in.columns:
    df_in['customer_city_norm'] = (
        df_in['customer_city'].str.strip().str.lower()
    )
    print(f"    ✅ customer_city_norm creada")

if 'customer_state' in df_in.columns:
    df_in['customer_state_norm'] = (
        df_in['customer_state'].str.strip().str.lower()
    )
    print(f"    ✅ customer_state_norm creada")

# ------------------------------------------------------------
# 7.6 CÓDIGO POSTAL REAL (a partir del state)
# ------------------------------------------------------------
# 📮 El postal_code original del CSV de seguros es aleatorio: no
#    tiene relación con la ciudad ni con el estado (por eso solo
#    ~400 hacían merge con Alemania). Aquí lo reemplazamos por un
#    CP REAL del catálogo de municipios, escogido dentro del mismo
#    estado (state) de la póliza. Así cada póliza queda con un CP
#    que existe en Alemania y que además pertenece a su estado.
print("\n" + "-" * 60)
print("[7.6] 📮 CÓDIGO POSTAL REAL A PARTIR DEL STATE")
print("-" * 60)

import os
import re

# 7.6.1 Estados del seguro → nombre oficial (igual que en la tabla de Alemania)
mapeo_estados = {
    "bavaria": "bayern", "bayern": "bayern",
    "saxony": "sachsen", "sachsen": "sachsen",
    "hesse": "hessen", "hessen": "hessen",
    "north rhine-westphalia": "nordrhein-westfalen", "nrw": "nordrhein-westfalen",
    "nordrhein-westfalen": "nordrhein-westfalen",
    "lower saxony": "niedersachsen", "niedersachsen": "niedersachsen",
    "rhineland-palatinate": "rheinland-pfalz", "rheinland-pfalz": "rheinland-pfalz",
    "thuringia": "thüringen", "thuringen": "thüringen", "thüringen": "thüringen",
    "saxony-anhalt": "sachsen-anhalt", "sachsen-anhalt": "sachsen-anhalt",
    "mecklenburg-western pomerania": "mecklenburg-vorpommern",
    "mecklenburg-vorpommern": "mecklenburg-vorpommern",
    "baden-wuerttemberg": "baden-württemberg", "baden-wurttemberg": "baden-württemberg",
    "baden-württemberg": "baden-württemberg",
    "schleswig-holstein": "schleswig-holstein", "brandenburg": "brandenburg",
    "berlin": "berlin", "hamburg": "hamburg", "bremen": "bremen", "saarland": "saarland",
}
col_state = "state" if "state" in df_in.columns else "customer_state"
df_in["customer_state_norm"] = (
    df_in[col_state].astype(str).str.strip().str.lower().map(mapeo_estados)
)
print(f"    Estados sin mapear: {df_in['customer_state_norm'].isna().sum()}")

# 7.6.2 Catálogo de CPs reales (tabla de Alemania ya limpia)
if os.path.exists("germany_municipalities_clean.csv"):
    cat = pd.read_csv("germany_municipalities_clean.csv", dtype=str,
                      keep_default_na=False)
elif "df_de" in globals():
    cat = df_de.copy()
else:
    raise FileNotFoundError(
        "❌ Falta germany_municipalities_clean.csv: ejecuta antes la celda de Alemania"
    )

cat["postal_code"] = (
    cat["postal_code"].astype(str).str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.replace(r"[^\d]", "", regex=True)
)
cat = cat[cat["postal_code"].str.len().between(1, 5)].copy()
cat["postal_code"] = cat["postal_code"].str.zfill(5)
cat["state"] = cat["state"].astype(str).str.strip().str.lower()
cat = cat[["postal_code", "state"]].drop_duplicates()
print(f"    Catálogo Alemania: {cat['postal_code'].nunique():,} CPs únicos, "
      f"{cat['state'].nunique()} estados")

# Estados del seguro que no existen en el catálogo (revisar el mapeo)
faltan = set(df_in["customer_state_norm"].dropna()) - set(cat["state"])
if faltan:
    raise ValueError(f"❌ Estados del seguro sin equivalente en Alemania: {faltan}")

# 7.6.3 CPs de cada estado. Se excluyen los CPs que están en 2 estados
#       a la vez, para que el CP siempre apunte al estado correcto.
estados_por_cp = cat.groupby("postal_code")["state"].nunique()
cps_exclusivos = set(estados_por_cp[estados_por_cp == 1].index)

pool_estado = {}
for est, g in cat.groupby("state"):
    cps = sorted(g["postal_code"].unique())
    excl = [cp for cp in cps if cp in cps_exclusivos]
    pool_estado[est] = excl if excl else cps
pool_nacional = sorted(cat["postal_code"].unique())

# 7.6.4 Asignar 1 CP real por póliza (misma póliza → mismo CP).
#       Semilla fija para que el resultado sea reproducible.
rng = np.random.default_rng(42)
pol = df_in[["policy_id", "customer_state_norm"]].drop_duplicates("policy_id")

cp_por_poliza = {}
origen_por_poliza = {}
for est, g in pol.groupby("customer_state_norm", dropna=False):
    ids = g["policy_id"].tolist()
    if pd.notna(est) and est in pool_estado:
        pool, origen = pool_estado[est], "state"
    else:
        pool, origen = pool_nacional, "fallback_nacional"
    # sin repetir CP mientras alcancen; si faltan, se repite
    elegidos = rng.choice(pool, size=len(ids), replace=len(ids) > len(pool))
    for pid, cp in zip(ids, elegidos):
        cp_por_poliza[pid] = cp
        origen_por_poliza[pid] = origen

df_in["postal_code_original"] = df_in["postal_code"]
df_in["postal_code"] = df_in["policy_id"].map(cp_por_poliza)
df_in["postal_code_source"] = df_in["policy_id"].map(origen_por_poliza)

# 7.6.5 Validación: 100% de los CPs deben existir en Alemania
cps_catalogo = set(cat["postal_code"])
n_ok = df_in["postal_code"].isin(cps_catalogo).sum()
print(f"    ✅ CPs reales asignados: {n_ok:,} / {len(df_in):,} "
      f"({n_ok / len(df_in):.1%} existen en Alemania)")
print(f"    Origen: {df_in['postal_code_source'].value_counts().to_dict()}")

estado_del_cp = cat[cat["postal_code"].isin(cps_exclusivos)].set_index("postal_code")["state"]
coincide = (df_in["postal_code"].map(estado_del_cp) == df_in["customer_state_norm"])
print(f"    🗺️ CP pertenece al state de la póliza: {coincide.mean():.1%}")

if n_ok != len(df_in):
    raise ValueError("❌ Hay CPs que no existen en el catálogo de Alemania")

# ------------------------------------------------------------
# 7.9 RE-VALIDACIÓN DE NEGATIVOS
# ------------------------------------------------------------
print("\n" + "-" * 60)
print("[7.9] 🔁 RE-VALIDACIÓN DE NEGATIVOS")
print("-" * 60)

for col in montos_positivos:
    if col not in df_in.columns:
        continue
    df_in[col] = pd.to_numeric(df_in[col], errors="coerce").fillna(0)
    n_neg = (df_in[col] < 0).sum()
    print(f"    {'⚠️ ' if n_neg > 0 else '✅'} {col:<32} negativos={n_neg}")

# ============================================================
# 🩹 [7.96] CORRECCIONES CRÍTICAS (PRE-MERGE)
# ============================================================
print("\n" + "=" * 60)
print("🩹 [7.96] CORRECCIONES CRÍTICAS DE CALIDAD")
print("=" * 60)

# ------------------------------------------------------------
# 7.96.1 DUPLICADOS EN policy_id
# ------------------------------------------------------------
print("\n[7.96.1] 🔑 Análisis de duplicados en policy_id")

n_dup = df_in.duplicated(subset=['policy_id'], keep=False).sum()
n_pol_dup = df_in.loc[df_in.duplicated(subset=['policy_id'], keep=False),
                      'policy_id'].nunique()
print(f"    Filas con policy_id duplicado: {n_dup:,}")
print(f"    Pólizas únicas afectadas:      {n_pol_dup:,}")

dup_counts = df_in.groupby('policy_id').size().rename('rows_per_policy')
df_in = df_in.merge(dup_counts, on='policy_id', how='left')
df_in['has_multiple_claims'] = df_in['rows_per_policy'] > 1

if n_dup > 0:
    dup_check = df_in[df_in['has_multiple_claims']].groupby('policy_id').agg(
        n_rows=('policy_id', 'size'),
        n_claim_ids=('claim_id', 'nunique'),
        n_customers=('customer_id', 'nunique'),
    )
    dup_check['tipo'] = np.where(
        dup_check['n_claim_ids'] > 1, 'multi_siniestro',
        np.where(dup_check['n_customers'] > 1, 'merge_roto', 'duplicado_exacto')
    )
    print(f"    Desglose:")
    print(dup_check['tipo'].value_counts().to_string())

    mask_dup_exacto = df_in.duplicated(
        subset=['policy_id', 'claim_id', 'customer_id'], keep='first'
    )
    n_exact = mask_dup_exacto.sum()
    if n_exact > 0:
        df_in = df_in[~mask_dup_exacto].copy()
        print(f"    ✂️  {n_exact:,} duplicados exactos eliminados")

# ------------------------------------------------------------
# 7.96.2 INCOHERENCIAS DE MONTOS EN SINIESTROS
# ------------------------------------------------------------
print("\n[7.96.2] 🔗 Incoherencias entre montos de siniestros")

con_claim = df_in['claim_id'].notna() & (df_in['claim_id'] != 'Sin Siniestro')

# Regla 1: Aprobado > Reclamado o Settlement > Aprobado
mask_incoh_1 = con_claim & (
    (df_in['claim_approved_amount_eur'] > df_in['claim_amount_eur']) |
    (df_in['settlement_amount_eur'] > df_in['claim_approved_amount_eur'])
)

# Regla 2: Reclamado=0 pero Aprobado>0
mask_incoh_2 = con_claim & (
    (df_in['claim_amount_eur'] == 0) &
    (df_in['claim_approved_amount_eur'] > 0)
)

mask_incoherente = mask_incoh_1 | mask_incoh_2
n_incoh = mask_incoherente.sum()
print(f"    Regla 1 (Aprobado>Reclamado / Settlement>Aprobado): {mask_incoh_1.sum():,}")
print(f"    Regla 2 (Reclamado=0 pero Aprobado>0):              {mask_incoh_2.sum():,}")
print(f"    Total incoherentes:                                 {n_incoh:,} "
      f"({safe_pct(n_incoh, len(df_in)):.1%})")

df_in['claim_amounts_inconsistent'] = mask_incoherente

cols_montos_claim = ['claim_amount_eur',
                     'claim_approved_amount_eur',
                     'settlement_amount_eur']

for c in cols_montos_claim:
    df_in[f'{c}_raw'] = df_in[c]

# Los montos NO se anulan: se conservan y quedan marcados en
# claim_amounts_inconsistent para poder analizar con y sin esas filas.
print(f"    🚩 {n_incoh:,} filas marcadas como incoherentes (montos conservados)")

# ------------------------------------------------------------
# 7.96.3 WINSORIZACIÓN DE OUTLIERS
# ------------------------------------------------------------
print("\n[7.96.3] 📏 Winsorización de outliers (p99)")

# Solo sobre pólizas con siniestro: los 0 de "sin siniestro" no deben
# arrastrar los percentiles ni subirse al p01.
for col in cols_montos_claim:
    serie_valida = df_in.loc[con_claim, col].dropna()
    if len(serie_valida) == 0:
        continue
    p99 = serie_valida.quantile(0.99)
    p01 = serie_valida.quantile(0.01)
    n_high = (serie_valida > p99).sum()
    n_low  = (serie_valida < p01).sum()
    df_in.loc[con_claim, col] = df_in.loc[con_claim, col].clip(lower=p01, upper=p99)
    print(f"    {col:<28} p01={p01:>12,.2f}  p99={p99:>14,.2f}  "
          f"(out: {n_low}L/{n_high}H)")

for col in ['premium_amount_eur', 'customer_lifetime_value_eur',
            'coverage_limit_eur']:
    if col not in df_in.columns:
        continue
    p99 = df_in[col].quantile(0.99)
    n_out = (df_in[col] > p99).sum()
    if n_out > 0:
        df_in[col] = df_in[col].clip(upper=p99)
        print(f"    {col:<28} p99={p99:>14,.2f}  (out: {n_out}H)")

# ------------------------------------------------------------
# 7.96.4 NEGATIVOS → NaN (no abs)
# ------------------------------------------------------------
print("\n[7.96.4] 🚫 Negativos → NaN")

cols_check_neg = [
    "premium_amount_eur", "claim_amount_eur",
    "claim_approved_amount_eur", "settlement_amount_eur",
    "coverage_limit_eur", "total_paid_eur",
    "customer_lifetime_value_eur",
]
for col in cols_check_neg:
    if col not in df_in.columns:
        continue
    n_neg = (pd.to_numeric(df_in[col], errors='coerce') < 0).sum()
    if n_neg > 0:
        df_in.loc[df_in[col] < 0, col] = np.nan
        print(f"    🔧 {col:<28} → {n_neg} negativos → NaN")
    else:
        print(f"    ✅ {col:<28} sin negativos")

for col in ['premium_amount_eur', 'coverage_limit_eur']:
    if col in df_in.columns:
        n_nan = df_in[col].isna().sum()
        if n_nan > 0:
            med = df_in[col].median()
            df_in[col] = df_in[col].fillna(med)
            print(f"    🔁 {col:<28} {n_nan} NaN → mediana {med:,.2f}")

# ------------------------------------------------------------
# 7.96.5 RE-VALIDACIÓN
# ------------------------------------------------------------
print("\n[7.96.5] 🔍 Re-validación de coherencia")

con_claim = df_in['claim_id'].notna() & (df_in['claim_id'] != 'Sin Siniestro')
mask_ok = (con_claim & ~df_in['claim_amounts_inconsistent']
           & df_in[cols_montos_claim].notna().all(axis=1))

n_ap = (mask_ok & (df_in['claim_approved_amount_eur'] >
                   df_in['claim_amount_eur'])).sum()
n_se = (mask_ok & (df_in['settlement_amount_eur'] >
                   df_in['claim_approved_amount_eur'])).sum()
print(f"    Filas coherentes con siniestro: {mask_ok.sum():,}")
print(f"    Aprobado > Reclamado:  {n_ap:,}")
print(f"    Settlement > Aprobado: {n_se:,}")
if n_ap == 0 and n_se == 0:
    print("    ✅ Coherencia asegurada en las filas no marcadas")
print(f"    🚩 Filas incoherentes conservadas y marcadas: "
      f"{df_in['claim_amounts_inconsistent'].sum():,}")

# ============================================================
# 🆕 TRANSFORMACIÓN ANALÍTICA
# ============================================================
print("\n" + "=" * 60)
print("🆕 TRANSFORMACIÓN ANALÍTICA")
print("=" * 60)

# ------------------------------------------------------------
# 8. EDAD desde date_of_birth
# ------------------------------------------------------------
today = pd.Timestamp.today()
if 'date_of_birth' in df_in.columns:
    age_calc = ((today - df_in['date_of_birth']).dt.days / 365.25).round(0)
    df_in['age'] = age_calc.fillna(df_in['age'])

# ------------------------------------------------------------
# 9. VARIABLES DERIVADAS
# ------------------------------------------------------------
print("\n[9] 🧮 Variables derivadas...")

df_in['policy_duration_days'] = (
    df_in['policy_end_date'] - df_in['policy_start_date']
).dt.days
df_in['policy_age_years'] = ((today - df_in['policy_start_date']).dt.days / 365.25)
df_in['is_active'] = df_in['policy_status'] == 'Active'
df_in['is_churned'] = df_in['policy_status'].isin(['Cancelled', 'Lapsed', 'Expired'])

# claim_ratio solo donde tiene sentido: póliza con siniestro, monto reclamado > 0
# y montos coherentes. En el resto queda NaN (no 0: un 0 sería "se aprobó nada").
_con_siniestro = df_in['claim_id'].notna() & (df_in['claim_id'] != 'Sin Siniestro')
_mask_ratio = (_con_siniestro
               & ~df_in['claim_amounts_inconsistent']
               & (df_in['claim_amount_eur'] > 0))
_ratio = df_in['claim_approved_amount_eur'] / df_in['claim_amount_eur'].where(
    df_in['claim_amount_eur'] > 0)
df_in['claim_ratio'] = _ratio.where(_mask_ratio)

# En [6.9] las fechas de siniestro se rellenaron con "Sin Fecha" (quedaron
# como texto). errors="coerce" vuelve a convertir esos textos en NaT para
# poder restar fechas sin error.
claim_dt      = pd.to_datetime(df_in['claim_date'], errors='coerce')
settlement_dt = pd.to_datetime(df_in['settlement_date'], errors='coerce')
adjustment_dt = pd.to_datetime(df_in['adjustment_date'], errors='coerce')

df_in['claim_settlement_days'] = (settlement_dt - claim_dt).dt.days
df_in['claim_adjustment_days'] = (adjustment_dt - claim_dt).dt.days
df_in['premium_per_coverage'] = np.where(
    df_in['coverage_limit_eur'] > 0,
    df_in['premium_amount_eur'] / df_in['coverage_limit_eur'],
    np.nan
)
df_in['discount_total_pct'] = (
    df_in['discount_applied_pct'] +
    df_in['no_claims_bonus_pct'] +
    df_in['loyalty_discount_pct'] +
    df_in['multi_policy_discount_pct']
)
df_in['has_claim'] = df_in['claim_id'].notna() & (df_in['claim_id'] != 'Sin Siniestro')
df_in['has_broker'] = df_in['broker_id'].notna() & (df_in['broker_id'] != 'Sin Broker')
df_in['is_fraud'] = df_in['fraud_flag'] == True
df_in['is_denied'] = df_in['claim_status'].isin(['Denied', 'Rejected'])
df_in['is_paid'] = df_in['claim_status'] == 'Paid'
df_in['is_pending'] = df_in['claim_status'].isin(['Open', 'In Progress', 'Under Review'])
df_in['complaint_flag'] = df_in['complaint_filed'] == True

# ------------------------------------------------------------
# 10. BANDAS / SEGMENTOS (CORREGIDO)
# ------------------------------------------------------------
print("\n[10] 📊 Bandas y segmentos...")

def crear_banda(serie, bins, labels, nombre):
    """
    Crea una banda con pd.cut validando que labels = len(bins) - 1.
    """
    if len(labels) != len(bins) - 1:
        raise ValueError(
            f"[{nombre}] labels={len(labels)} debe ser bins-1={len(bins)-1}"
        )
    try:
        return pd.cut(serie, bins=bins, labels=labels,
                      include_lowest=True)
    except Exception as e:
        print(f"    ⚠️  {nombre}: {e}")
        return pd.Series([np.nan] * len(serie), index=serie.index)

def crear_clv_band(serie, n_cuartiles=4):
    """
    Crea bandas de CLV por cuantiles. Maneja el caso de cuantiles
    duplicados o valores todos iguales.
    """
    s = pd.to_numeric(serie, errors='coerce')
    s_valid = s.dropna()

    if len(s_valid) == 0:
        return pd.Series([np.nan] * len(serie), index=serie.index)

    # Intento 1: qcut con duplicates='drop'
    try:
        return pd.qcut(
            s, q=n_cuartiles,
            labels=['Bajo', 'Medio', 'Alto', 'Premium'],
            duplicates='drop'
        )
    except (ValueError, IndexError):
        pass

    # Intento 2: cuantiles únicos
    qs = s_valid.quantile([0, .25, .5, .75, 1]).values
    qs_unicos = np.unique(qs)

    if len(qs_unicos) < 2:
        # Todos los valores son iguales → una sola categoría
        return pd.Series(['Medio'] * len(serie), index=serie.index)

    bins = [-np.inf] + list(qs_unicos[1:-1]) + [np.inf]
    bins = sorted(set(bins))

    n_intervalos = len(bins) - 1
    labels_full = ['Bajo', 'Medio', 'Alto', 'Premium']
    if n_intervalos <= len(labels_full):
        labels = labels_full[:n_intervalos]
    else:
        labels = [f'Q{i+1}' for i in range(n_intervalos)]

    return pd.cut(s, bins=bins, labels=labels, include_lowest=True)

# satisfaction_band (escala 0-5)
df_in['satisfaction_band'] = crear_banda(
    df_in['customer_satisfaction_score'],
    bins=[0, 2, 3, 4, 5.01],
    labels=['Muy Baja', 'Baja', 'Media', 'Alta'],
    nombre='satisfaction_band'
)

# nps_band (escala -100 a 100)
df_in['nps_band'] = crear_banda(
    df_in['nps_score'],
    bins=[-100.01, -50, 0, 50, 100.01],
    labels=['Detractor', 'Passive', 'Promoter', 'Promoter Elite'],
    nombre='nps_band'
)

# fraud_risk_band
df_in['fraud_risk_band'] = crear_banda(
    df_in['fraud_score'],
    bins=[0, 25, 50, 75, 100.01],
    labels=['Bajo', 'Medio', 'Alto', 'Muy Alto'],
    nombre='fraud_risk_band'
)

# risk_band
df_in['risk_band'] = crear_banda(
    df_in['risk_score'],
    bins=[0, 25, 50, 75, 100.01],
    labels=['Bajo', 'Medio', 'Alto', 'Muy Alto'],
    nombre='risk_band'
)

# renewal_band
df_in['renewal_band'] = crear_banda(
    df_in['renewal_probability_pct'],
    bins=[0, 25, 50, 75, 100.01],
    labels=['Muy Baja', 'Baja', 'Media', 'Alta'],
    nombre='renewal_band'
)

# tenure_band
df_in['tenure_band'] = crear_banda(
    df_in['customer_tenure_months'],
    bins=[0, 12, 36, 60, 120, 10000],
    labels=['Nuevo', 'Medio', 'Leal', 'VIP', 'Veterano'],
    nombre='tenure_band'
)

# clv_band (robusto)
df_in['clv_band'] = crear_clv_band(df_in['customer_lifetime_value_eur'])

print("    ✅ Bandas creadas:")
for b in ['satisfaction_band', 'nps_band', 'fraud_risk_band',
          'risk_band', 'renewal_band', 'tenure_band', 'clv_band']:
    if b in df_in.columns:
        n_valid = df_in[b].notna().sum()
        n_unicos = df_in[b].nunique(dropna=True)
        print(f"       {b:<22} → {n_valid:,} válidos, "
              f"{n_unicos} categorías")

# ------------------------------------------------------------
# 11. TEMPORALES
# ------------------------------------------------------------
print("\n[11] 📅 Variables temporales...")

df_in['claim_year'] = claim_dt.dt.year
df_in['claim_month'] = claim_dt.dt.month
df_in['policy_year'] = df_in['policy_start_date'].dt.year
df_in['renewal_year'] = df_in['renewal_date'].dt.year
df_in['policy_month'] = df_in['policy_start_date'].dt.month
df_in['policy_year_month'] = df_in['policy_start_date'].dt.to_period('M').astype(str)

# ------------------------------------------------------------
# 12. CHURN SCORE (corregido)
# ------------------------------------------------------------
print("\n[12] 🎯 Churn risk score (corregido)")

comp_sat = 1 - (df_in['customer_satisfaction_score'] / 5)
comp_ren = 1 - (df_in['renewal_probability_pct'] / 100)
comp_cmp = df_in['complaint_flag'].astype(float)
comp_chu = (df_in['churn_risk'] == 'High').astype(float)
comp_nps = 1 - ((df_in['nps_score'] + 100) / 200)

componentes = pd.DataFrame({
    'sat': comp_sat, 'ren': comp_ren, 'cmp': comp_cmp,
    'chu': comp_chu, 'nps': comp_nps,
})
pesos = pd.Series({'sat': 0.30, 'ren': 0.20, 'cmp': 0.20,
                   'chu': 0.15, 'nps': 0.15})

numerador   = (componentes * pesos).sum(axis=1, skipna=True)
denominador = (componentes.notna() * pesos).sum(axis=1)
# Evitar división por cero
denominador = denominador.replace(0, np.nan)
df_in['churn_risk_score'] = (numerador / denominador).clip(0, 1)
df_in['churn_components_available'] = componentes.notna().sum(axis=1)

df_in['churn_risk_segment'] = pd.cut(
    df_in['churn_risk_score'],
    bins=[-0.01, 0.3, 0.5, 0.7, 1.01],
    labels=['Bajo', 'Medio', 'Alto', 'Crítico']
)

churn_min = df_in['churn_risk_score'].min()
churn_max = df_in['churn_risk_score'].max()
print(f"    churn_risk_score: min={churn_min:.3f}  max={churn_max:.3f}")

# ============================================================
# 13. AUDITORÍA FINAL
# ============================================================
print("\n" + "-" * 60)
print("🧾 AUDITORÍA FINAL")
print("-" * 60)

print(f"Filas finales:    {len(df_in):,}")
print(f"Columnas finales: {df_in.shape[1]}")

n_dup = df_in.duplicated(subset=['policy_id']).sum()
n_pol_multi = df_in.loc[df_in['has_multiple_claims'], 'policy_id'].nunique()
print(f"\n🔑 Duplicados policy_id: {n_dup:,}")
print(f"   Pólizas multi-siniestro: {n_pol_multi:,}")
print(f"   Grano: póliza-siniestro")

print(f"\n🔍 Nulos restantes:")
nulos = df_in.isnull().sum().sort_values(ascending=False)
nulos_pos = nulos[nulos > 0]
if len(nulos_pos) == 0:
    print("    ✅ Sin nulos")
else:
    print(nulos_pos.head(10).to_string())

print(f"\n💸 Validación de montos:")
hay_negativos = False
for col in [c for c in df_in.columns
            if c.endswith("_eur") and not c.endswith("_raw")]:
    serie = pd.to_numeric(df_in[col], errors="coerce")
    n_neg = (serie < 0).sum()
    flag = "❌" if n_neg > 0 else "✅"
    if n_neg > 0:
        hay_negativos = True
    print(f"    {flag} {col:<32} neg={n_neg:>3}  "
          f"min={serie.min():>14,.2f}  max={serie.max():>14,.2f}")

if hay_negativos:
    raise ValueError("❌ Hay negativos tras limpieza")

print(f"\n🔗 Coherencia siniestros:")
con_claim = df_in['claim_id'].notna() & (df_in['claim_id'] != 'Sin Siniestro')
mask_ok = (con_claim & ~df_in['claim_amounts_inconsistent']
           & df_in[cols_montos_claim].notna().all(axis=1))
n_ap = (mask_ok & (df_in['claim_approved_amount_eur'] >
                   df_in['claim_amount_eur'])).sum()
n_se = (mask_ok & (df_in['settlement_amount_eur'] >
                   df_in['claim_approved_amount_eur'])).sum()
print(f"    Aprobado > Reclamado:  {n_ap:,}")
print(f"    Settlement > Aprobado: {n_se:,}")
print(f"    Filas marcadas inconsistentes: "
      f"{df_in['claim_amounts_inconsistent'].sum():,}")

print(f"\n🆕 Variables derivadas clave:")
for v in ['has_multiple_claims', 'rows_per_policy',
          'claim_amounts_inconsistent', 'survey_responded',
          'customer_city_norm', 'customer_state_norm',
          'churn_risk_score', 'churn_risk_segment',
          'satisfaction_band', 'nps_band', 'risk_band', 'renewal_band',
          'tenure_band', 'clv_band']:
    if v in df_in.columns:
        print(f"    ✅ {v}")

# ============================================================
# 14. GUARDAR
# ============================================================
df_in.to_csv("insurance_life_only_clean.csv", index=False, encoding="utf-8")
print(f"\n✅ Guardado: insurance_life_only_clean.csv ({len(df_in)} filas)")
df_in.to_csv("insurance_life_geo_ready.csv", index=False, encoding="utf-8")
print(f"✅ Guardado: insurance_life_geo_ready.csv ({len(df_in)} filas) → listo para el merge")
print("=" * 60)

LIMPIEZA PRE-MERGE: insurance_policies_claims

[1] 📥 Cargado: 52676 filas x 76 columnas
    ✅ DataFrame no vacío: 52,676 filas

[2] ✂️ Columnas eliminadas: ['customer_email']

------------------------------------------------------------
[2.5] 🩺 DIAGNÓSTICO DE NULOS Y VACÍOS
------------------------------------------------------------
                            tipo_dato  vacios_str  total_problemas  pct_del_total
property_type                  object       31992            31992          60.73
broker_id                      object       30825            30825          58.52
property_year_built            object       27281            27281          51.79
nps_score                      object       25252            25252          47.94
claim_id                       object       25143            25143          47.73
vehicle_make                   object       24071            24071          45.70
customer_satisfaction_score    object       22753            22753          43.19
vehicle_

/tmp/ipykernel_893/662840526.py:366: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_in["fraud_flag"] = df_in["fraud_flag"].fillna(False).astype(bool)



------------------------------------------------------------
[7] 📊 EDAD: mediana por género
------------------------------------------------------------
    ✅ Edades nulas después: 0

------------------------------------------------------------
[7.5] 🔤 NORMALIZAR city/state PARA EL MERGE
------------------------------------------------------------

------------------------------------------------------------
[7.6] 📮 CÓDIGO POSTAL REAL A PARTIR DEL STATE
------------------------------------------------------------
    Estados sin mapear: 0
    Catálogo Alemania: 6,440 CPs únicos, 16 estados
    ✅ CPs reales asignados: 6,593 / 6,593 (100.0% existen en Alemania)
    Origen: {'state': 6593}
    🗺️ CP pertenece al state de la póliza: 100.0%

------------------------------------------------------------
[7.9] 🔁 RE-VALIDACIÓN DE NEGATIVOS
------------------------------------------------------------
    ⚠️  premium_amount_eur               negativos=19
    ✅ coverage_limit_eur               ne

In [38]:
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# ENRIQUECIMIENTO — Seguros + Municipios (merge sin duplicados)
# ------------------------------------------------------------
print("=" * 60)
print("🔗 ENRIQUECIMIENTO: Seguros + Alemania")
print("=" * 60)

# ------------------------------------------------------------
# 1. CARGA
# ------------------------------------------------------------
df_seguros = pd.read_csv("insurance_life_geo_ready.csv", dtype=str, keep_default_na=False)
df_alemania = pd.read_csv("germany_municipalities_clean.csv", dtype=str, keep_default_na=False)

print(f"📥 Seguros:  {df_seguros.shape[0]:,} filas x {df_seguros.shape[1]} columnas")
print(f"📥 Alemania: {df_alemania.shape[0]:,} filas x {df_alemania.shape[1]} columnas")

# ------------------------------------------------------------
# 2. NORMALIZAR LLAVE (postal_code)
# ------------------------------------------------------------
def normalizar_cp(s):
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"[^\d]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan})
        .str.zfill(5)
    )

df_seguros["postal_code"] = normalizar_cp(df_seguros["postal_code"])
df_alemania["postal_code"] = normalizar_cp(df_alemania["postal_code"])

# ------------------------------------------------------------
# 3. RENOMBRAR SEGUROS
# ------------------------------------------------------------
rename_seguros = {
    "city":    "customer_city",
    "address": "customer_address",
    "state":   "customer_state",
    "country": "customer_country",
}
rename_seguros = {k: v for k, v in rename_seguros.items() if k in df_seguros.columns}
df_seguros = df_seguros.rename(columns=rename_seguros)

# ------------------------------------------------------------
# 4. RENOMBRAR ALEMANIA
# ------------------------------------------------------------
rename_alemania = {
    "state":              "municipality_state",
    "district":           "municipality_district",
    "regional_type":      "municipality_regional_type",
    "municipality_name":  "municipality_name",
    "population_total":   "municipality_population",
    "population_density": "municipality_density",
    "male":               "municipality_male",
    "female":             "municipality_female",
    "latitude":           "municipality_latitude",
    "longitude":          "municipality_longitude",
}
rename_alemania = {k: v for k, v in rename_alemania.items() if k in df_alemania.columns}
df_alemania = df_alemania.rename(columns=rename_alemania)

# ------------------------------------------------------------
# 5. CONVERTIR NUMÉRICOS EN ALEMANIA
# ------------------------------------------------------------
cols_numericas = [
    "municipality_population", "municipality_density",
    "municipality_male", "municipality_female",
    "municipality_latitude", "municipality_longitude",
]
cols_numericas = [c for c in cols_numericas if c in df_alemania.columns]
for col in cols_numericas:
    df_alemania[col] = pd.to_numeric(
        df_alemania[col].astype(str).str.replace(",", ".", regex=False),
        errors="coerce",
    )

# ------------------------------------------------------------
# 6. AGREGAR ALEMANIA A 1 FILA POR CP  ⚠️ CLAVE PARA NO DUPLICAR
# ------------------------------------------------------------
# 📌 Alemania tiene varios municipios por CP. Si hacemos merge
#    directo, cada póliza se multiplica. Agregamos a 1 fila por
#    CP antes del merge para evitar el efecto "147% match".
print("\n🧮 Agregando Alemania a 1 fila por CP...")

agg_dict = {}
for col in ["municipality_name", "municipality_state",
            "municipality_district", "municipality_regional_type"]:
    if col in df_alemania.columns:
        agg_dict[col] = "first"
for col in ["municipality_population", "municipality_male", "municipality_female"]:
    if col in df_alemania.columns:
        agg_dict[col] = "sum"
for col in ["municipality_density", "municipality_latitude", "municipality_longitude"]:
    if col in df_alemania.columns:
        agg_dict[col] = "mean"

df_alemania_unico = df_alemania.groupby("postal_code", as_index=False).agg(agg_dict)
print(f"    Alemania: {len(df_alemania):,} → {len(df_alemania_unico):,} filas (1 por CP)")

# ------------------------------------------------------------
# 7. MERGE
# ------------------------------------------------------------
print("\n🔗 Haciendo merge...")
df_final = pd.merge(
    df_seguros,
    df_alemania_unico,
    on="postal_code",
    how="left",
    validate="many_to_one",
    indicator=True,
)

print(f"\n🚦 Auditoría del match:")
print(df_final["_merge"].value_counts().to_string())

match_counts = df_final["_merge"].value_counts()
total = len(df_final)
if "both" in match_counts:
    pct = match_counts["both"] / total * 100
    print(f"\n  ✅ Match exitoso: {match_counts['both']:,} filas ({pct:.2f}%)")
if "left_only" in match_counts:
    pct = match_counts["left_only"] / total * 100
    print(f"  ⚠️  Sin match: {match_counts['left_only']:,} filas ({pct:.2f}%)")

# ------------------------------------------------------------
# 8. RELLENAR NULOS INTRODUCIDOS POR EL MERGE
# ------------------------------------------------------------
# 📌 Las pólizas cuyo CP no existe en el catálogo alemán quedan
#    con nulos en las columnas de municipio. Las marcamos como
#    "Desconocido" para que no queden vacías.
print("\n🔧 Rellenando nulos introducidos por el merge...")
cols_municipio = [c for c in df_final.columns if c.startswith("municipality_")]
for col in cols_municipio:
    n = df_final[col].isnull().sum()
    if n > 0:
        if df_final[col].dtype == "object":
            df_final[col] = df_final[col].fillna("Desconocido")
        else:
            df_final[col] = df_final[col].fillna(0)
        print(f"    ✅ {col:<30} → {n} nulos rellenados")

# Quitar columna _merge
df_final = df_final.drop(columns=["_merge"])

# ------------------------------------------------------------
# 9. GUARDAR
# ------------------------------------------------------------
df_final.to_csv("insurance_life_enriched_v1.csv", index=False, encoding="utf-8")
print(f"\n✅ Guardado: insurance_life_enriched_v1.csv ({len(df_final):,} filas)")

# ------------------------------------------------------------
# 10. AUDITORÍA FINAL
# ------------------------------------------------------------
print("\n" + "=" * 60)
print("🧾 AUDITORÍA FINAL DEL ENRIQUECIDO")
print("=" * 60)
print(f"Filas: {len(df_final):,}")
print(f"Columnas: {df_final.shape[1]}")
print(f"Duplicados en policy_id: {df_final.duplicated(subset=['policy_id']).sum()}")

nulos = df_final.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
if len(nulos) == 0:
    print("\n✅ Sin nulos restantes")
else:
    print(f"\n⚠️  Nulos restantes en {len(nulos)} columnas:")
    print(nulos.to_string())

🔗 ENRIQUECIMIENTO: Seguros + Alemania
📥 Seguros:  6,512 filas x 110 columnas
📥 Alemania: 10,757 filas x 12 columnas

🧮 Agregando Alemania a 1 fila por CP...
    Alemania: 10,757 → 6,440 filas (1 por CP)

🔗 Haciendo merge...

🚦 Auditoría del match:
_merge
both          6512
left_only        0
right_only       0

  ✅ Match exitoso: 6,512 filas (100.00%)
  ⚠️  Sin match: 0 filas (0.00%)

🔧 Rellenando nulos introducidos por el merge...

✅ Guardado: insurance_life_enriched_v1.csv (6,512 filas)

🧾 AUDITORÍA FINAL DEL ENRIQUECIDO
Filas: 6,512
Columnas: 120
Duplicados en policy_id: 0

✅ Sin nulos restantes


In [39]:
import os
import numpy as np
import pandas as pd
from google.colab import files

# ------------------------------------------------------------
# RUTA DE SALIDA (visible desde el kernel de Colab)
# ------------------------------------------------------------
# Entradas: los CSV limpios que generan las celdas anteriores del notebook
SEGUROS_PATH = "/content/insurance_life_geo_ready.csv"
ALEMANIA_PATH = "/content/germany_municipalities_clean.csv"

OUTPUT_DIR = "/content"
OUTPUT_NAME = "insurance_life_enriched.csv"
OUTPUT_PATH = os.path.join(OUTPUT_DIR, OUTPUT_NAME)


# ------------------------------------------------------------
# LÓGICA DEL PIPELINE
# ------------------------------------------------------------
def normalizar_cp(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.replace(r"[^\d]", "", regex=True)
        .replace({"": np.nan, "nan": np.nan})
        .str.zfill(5)
    )


def ejecutar_pipeline(file_seguros_path: str, file_alemania_path: str) -> tuple[str, str]:
    logs: list[str] = []

    def log(msg: object = "") -> None:
        print(msg)
        logs.append(str(msg))

    log("=" * 60)
    log("🔗 ENRIQUECIMIENTO v3: Seguros + Alemania + Estados + groupby")
    log("=" * 60)

    # ------------------------------------------------------------
    # 1. CARGA
    # ------------------------------------------------------------
    df_seguros = pd.read_csv(file_seguros_path, dtype=str, keep_default_na=False)
    df_alemania = pd.read_csv(file_alemania_path, dtype=str, keep_default_na=False)

    log(f"📥 Seguros:  {df_seguros.shape[0]:,} x {df_seguros.shape[1]}")
    log(f"📥 Alemania: {df_alemania.shape[0]:,} x {df_alemania.shape[1]}")

    # ------------------------------------------------------------
    # 2. NORMALIZAR CP
    # ------------------------------------------------------------
    df_seguros["postal_code"] = normalizar_cp(df_seguros["postal_code"])
    df_alemania["postal_code"] = normalizar_cp(df_alemania["postal_code"])

    # ------------------------------------------------------------
    # 3. RENOMBRAR
    # ------------------------------------------------------------
    df_seguros = df_seguros.rename(columns={
        "city": "customer_city",
        "address": "customer_address",
        "state": "customer_state",
        "country": "customer_country",
    })

    df_alemania = df_alemania.rename(columns={
        "state":              "municipality_state",
        "district":           "municipality_district",
        "regional_type":      "municipality_regional_type",
        "municipality_name":  "municipality_name",
        "population_total":   "municipality_population",
        "population_density": "municipality_density",
        "male":               "municipality_male",
        "female":             "municipality_female",
        "latitude":           "municipality_latitude",
        "longitude":          "municipality_longitude",
    })

    # ------------------------------------------------------------
    # 4. NORMALIZAR ESTADOS DEL CLIENTE
    # ------------------------------------------------------------
    mapeo_estados = {
        "bavaria": "bayern", "saxony": "sachsen", "hesse": "hessen",
        "north rhine-westphalia": "nordrhein-westfalen", "nrw": "nordrhein-westfalen",
        "lower saxony": "niedersachsen", "rhineland-palatinate": "rheinland-pfalz",
        "thuringia": "thüringen", "saxony-anhalt": "sachsen-anhalt",
        "mecklenburg-western pomerania": "mecklenburg-vorpommern",
        "baden-wuerttemberg": "baden-württemberg", "baden-wurttemberg": "baden-württemberg",
        "bw": "baden-württemberg", "by": "bayern", "be": "berlin", "bb": "brandenburg",
        "hb": "bremen", "hh": "hamburg", "he": "hessen", "mv": "mecklenburg-vorpommern",
        "ni": "niedersachsen", "nw": "nordrhein-westfalen", "rp": "rheinland-pfalz",
        "sl": "saarland", "sn": "sachsen", "st": "sachsen-anhalt",
        "sh": "schleswig-holstein", "th": "thüringen",
        "bayern": "bayern", "sachsen": "sachsen", "hessen": "hessen",
        "nordrhein-westfalen": "nordrhein-westfalen", "niedersachsen": "niedersachsen",
        "baden-württemberg": "baden-württemberg", "rheinland-pfalz": "rheinland-pfalz",
        "schleswig-holstein": "schleswig-holstein", "brandenburg": "brandenburg",
        "berlin": "berlin", "hamburg": "hamburg", "bremen": "bremen",
        "thüringen": "thüringen", "thuringen": "thüringen",
        "sachsen-anhalt": "sachsen-anhalt", "mecklenburg-vorpommern": "mecklenburg-vorpommern",
        "saarland": "saarland",
        "n/a": None, "na": None, "-": None, "": None,
    }

    df_seguros["customer_state_norm"] = (
        df_seguros["customer_state"].str.lower().str.strip().map(mapeo_estados)
    )

    # ------------------------------------------------------------
    # 5. CONVERTIR NUMÉRICOS EN ALEMANIA
    # ------------------------------------------------------------
    for col in [
        "municipality_population", "municipality_density",
        "municipality_male", "municipality_female",
        "municipality_latitude", "municipality_longitude",
    ]:
        df_alemania[col] = pd.to_numeric(
            df_alemania[col].astype(str).str.replace(",", ".", regex=False),
            errors="coerce",
        )

    # ------------------------------------------------------------
    # 6. AGREGAR ALEMANIA A 1 FILA POR CP
    # ------------------------------------------------------------
    df_alemania_cp = df_alemania.groupby("postal_code", as_index=False).agg({
        "municipality_name":       "first",
        "municipality_state":      "first",
        "municipality_district":   "first",
        "municipality_population": "sum",
        "municipality_density":    "mean",
        "municipality_latitude":   "mean",
        "municipality_longitude":  "mean",
    })

    log(f"\n🧮 Alemania por CP: {len(df_alemania):,} → {len(df_alemania_cp):,} filas")

    # ------------------------------------------------------------
    # 7. MERGE POR CP
    # ------------------------------------------------------------
    df_seguros = pd.merge(
        df_seguros, df_alemania_cp,
        on="postal_code", how="left",
        validate="many_to_one",
        suffixes=("", "_cp"),
    )

    match_cp = df_seguros["municipality_name"].notna().sum()
    pct_cp = match_cp / len(df_seguros) * 100
    log(f"\n✅ Merge por CP: {match_cp:,} / {len(df_seguros):,} ({pct_cp:.1f}%)")

    # ------------------------------------------------------------
    # 8. MERGE POR ESTADO (plan B)
    # ------------------------------------------------------------
    df_alemania_estado = df_alemania.groupby("municipality_state", as_index=False).agg({
        "municipality_population": "sum",
        "municipality_density":    "mean",
    })

    df_seguros = pd.merge(
        df_seguros,
        df_alemania_estado.rename(columns={
            "municipality_state":      "state_key",
            "municipality_population": "state_population",
            "municipality_density":    "state_density",
        }),
        left_on="customer_state_norm", right_on="state_key",
        how="left", validate="many_to_one",
    )

    match_estado = df_seguros["state_key"].notna().sum()
    pct_estado = match_estado / len(df_seguros) * 100
    log(f"✅ Merge por estado: {match_estado:,} / {len(df_seguros):,} ({pct_estado:.1f}%)")

    # ------------------------------------------------------------
    # 9. AGREGAR POR policy_id
    # ------------------------------------------------------------
    cols_agg: dict[str, str] = {}
    for c in df_seguros.columns:
        if c == "policy_id":
            continue
        if c in ("claim_amount_eur", "claim_approved_amount_eur", "settlement_amount_eur"):
            cols_agg[c] = "sum"
        elif c in ("fraud_score", "risk_score"):
            cols_agg[c] = "max"
        else:
            cols_agg[c] = "first"

    df_seguros["_n_claims"] = 1
    cols_agg["_n_claims"] = "sum"

    df_final = df_seguros.groupby("policy_id", as_index=False).agg(cols_agg)
    df_final = df_final.rename(columns={"_n_claims": "num_claims_detail"})

    log(f"\n🔽 Agrupado: {len(df_seguros):,} → {len(df_final):,} filas (1 por policy_id)")

    # ------------------------------------------------------------
    # 10. COMBINAR municipio ↔ estado
    # ------------------------------------------------------------
    df_final["municipality_population"] = df_final["municipality_population"].fillna(
        df_final["state_population"]
    )
    df_final["municipality_density"] = df_final["municipality_density"].fillna(
        df_final["state_density"]
    )
    df_final = df_final.drop(columns=["state_population", "state_density", "state_key"])

    # ------------------------------------------------------------
    # 11. RELLENO DE NULOS
    # ------------------------------------------------------------
    log("\n🔧 Rellenando nulos introducidos por los merges...")

    cols_geo_texto = [
        "municipality_name", "municipality_state", "municipality_district",
        "municipality_regional_type",
    ]
    for col in cols_geo_texto:
        if col in df_final.columns:
            n = df_final[col].isnull().sum()
            if n > 0:
                df_final[col] = df_final[col].fillna("Desconocido")
                log(f"    ✅ {col:<30} → 'Desconocido'  ({n} filas)")

    cols_geo_num = [
        "municipality_population", "municipality_density",
        "municipality_male", "municipality_female",
        "municipality_latitude", "municipality_longitude",
    ]
    for col in cols_geo_num:
        if col in df_final.columns:
            n = df_final[col].isnull().sum()
            if n > 0:
                df_final[col] = df_final[col].fillna(0)
                log(f"    ✅ {col:<30} → 0  ({n} filas)")

    for col in df_final.select_dtypes(include=["object"]).columns:
        n = df_final[col].isnull().sum()
        if n > 0:
            df_final[col] = df_final[col].fillna("Desconocido")
            log(f"    ✅ {col:<30} → 'Desconocido'  ({n} filas)")

    for col in df_final.select_dtypes(include=[np.number]).columns:
        n = df_final[col].isnull().sum()
        if n > 0:
            df_final[col] = df_final[col].fillna(0)
            log(f"    ✅ {col:<30} → 0  ({n} filas)")

    # ------------------------------------------------------------
    # 12. VERIFICACIÓN FINAL
    # ------------------------------------------------------------
    log("\n" + "=" * 60)
    log("🧾 VERIFICACIÓN FINAL")
    log("=" * 60)
    log(f"Filas: {len(df_final):,}")
    log(f"policy_id únicos: {df_final['policy_id'].nunique():,}")
    log(f"Duplicados: {df_final['policy_id'].duplicated().sum()}")

    assert df_final["policy_id"].duplicated().sum() == 0, "❌ Duplicados"

    nulos = df_final.isnull().sum()
    nulos = nulos[nulos > 0].sort_values(ascending=False)
    if len(nulos) == 0:
        log("\n✅ Sin nulos restantes — 0 nulos en TODAS las columnas")
    else:
        log("\n⚠️  Nulos restantes:")
        log(nulos.to_string())

    # ------------------------------------------------------------
    # 13. GUARDAR
    # ------------------------------------------------------------
    df_final.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    log(f"\n✅ Guardado: {OUTPUT_PATH} ({len(df_final):,} filas)")

    return "\n".join(logs), OUTPUT_PATH


# ------------------------------------------------------------
# EJECUCIÓN Y DESCARGA
# ------------------------------------------------------------
_, ruta_salida = ejecutar_pipeline(SEGUROS_PATH, ALEMANIA_PATH)

print("\n📥 Iniciando descarga...")
files.download(ruta_salida)

🔗 ENRIQUECIMIENTO v3: Seguros + Alemania + Estados + groupby
📥 Seguros:  6,512 x 110
📥 Alemania: 10,757 x 12

🧮 Alemania por CP: 10,757 → 6,440 filas

✅ Merge por CP: 6,512 / 6,512 (100.0%)
✅ Merge por estado: 6,512 / 6,512 (100.0%)

🔽 Agrupado: 6,512 → 6,512 filas (1 por policy_id)

🔧 Rellenando nulos introducidos por los merges...

🧾 VERIFICACIÓN FINAL
Filas: 6,512
policy_id únicos: 6,512
Duplicados: 0

✅ Sin nulos restantes — 0 nulos en TODAS las columnas

✅ Guardado: /content/insurance_life_enriched.csv (6,512 filas)

📥 Iniciando descarga...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>